# 05 · Deep learning on the raw time series (GPU)

Notebook 04 used **hand-made features** (z-scores, ratios). Here a **1D convolutional neural network (CNN)** reads each service's raw metric series directly and learns its own features.

**Input** per (case, service): 8 metric types × 144 time steps (24 minutes at 10-second resolution, fault injected in the middle). Each series is standardised with the median/IQR of its own normal window (same rule as notebook 02), plus 8 channels saying which metric types exist for this service.

**Model:** the same small CNN scores every service; inside a case, the scores go through a **softmax** ("which one of these is the culprit?") and the loss is the cross-entropy with the true culprit. This is called a *listwise* ranking loss. Because the network is shared across services, it learns what a *culprit-like pattern* looks like, never the service names.

**Evaluation:** exactly the same grouped 5-fold splits as notebook 04 (seed 0), plus leave-one-system-out.

> With only 270 cases a neural network can easily over-fit; the comparison with XGBoost is the point of this notebook, not a guaranteed win.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*(mismatched devices|pandas version 4).*")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
RESULTS = ROOT / "reports" / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

import torch, time
from rcalens.data import case_index
from rcalens.dl import build_dataset, train_cnn, score_cases
from rcalens.evaluate import evaluate, true_ranks, bootstrap_diff, grouped_folds, leave_one_system_out
idx = case_index()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
t = time.time(); data = build_dataset(idx); print(f"tensors for {len(data)} cases built in {time.time() - t:.0f}s")

device: cuda NVIDIA A100-PCIE-40GB


tensors for 270 cases built in 26s


In [2]:
import os
EPOCHS = int(os.environ.get("RCA_EPOCHS", 60))   # set RCA_EPOCHS=3 for a quick smoke test
SEEDS = (0, 1, 2)                                # 3 networks per fold, averaged (an "ensemble" reduces variance)
sys_of = idx.set_index("case")["system"]
def run(train_cases, test_cases):
    scores = None
    for s in SEEDS:
        m = train_cnn(data, list(train_cases), DEVICE, epochs=EPOCHS, seed=s)
        o = score_cases(m, data, list(test_cases), DEVICE)
        scores = o if scores is None else scores.assign(score=scores["score"] + o["score"])
    scores["system"] = scores["case"].map(sys_of)
    return scores

t = time.time()
cnn_cv = pd.concat([run(tr, te) for tr, te in grouped_folds(idx, n_splits=5, seed=0)])
print(f"grouped CV done in {time.time() - t:.0f}s")
evaluate(cnn_cv).round(3)

grouped CV done in 1021s


,AC@1,AC@3,AC@5,Avg@5,MRR
all,0.930,0.981,1.0,0.976,0.959
ob,0.856,0.978,1.0,0.958,0.919
ss,0.956,0.967,1.0,0.976,0.969
tt,0.978,1.000,1.0,0.996,0.989


In [3]:
xgb = pd.read_parquet(ROOT / "data" / "processed" / "oof_xgb.parquet")
rule = pd.read_parquet(ROOT / "data" / "processed" / "baseline_scores.parquet")
pd.concat({"rule": evaluate(rule).loc["all"], "XGBoost ranker": evaluate(xgb).loc["all"], "1D CNN": evaluate(cnn_cv).loc["all"]}, axis=1).T.round(3)

,AC@1,AC@3,AC@5,Avg@5,MRR
rule,0.707,0.926,0.963,0.882,0.820
XGBoost ranker,0.941,0.996,1.000,0.985,0.969
1D CNN,0.930,0.981,1.000,0.976,0.959


In [4]:
r_cnn, r_xgb = true_ranks(cnn_cv), true_ranks(xgb)
pd.DataFrame({m: bootstrap_diff(r_cnn, r_xgb, m) for m in ["AC@1", "Avg@5", "MRR"]}, index=["2.5%", "median", "97.5%"]).T.rename_axis("CNN − XGBoost")

,2.5%,median,97.5%
CNN − XGBoost,,,
AC@1,-0.037,-0.011,0.015
Avg@5,-0.017,-0.009,-0.001
MRR,-0.024,-0.010,0.003


### Ensemble: average the two model types
Scores are on different scales, so each case's scores are turned into within-case ranks first, then averaged.

In [5]:
ens = xgb[["case", "service", "system", "is_rc", "score"]].rename(columns={"score": "score_xgb"})
ens = ens.merge(cnn_cv[["case", "service", "score"]].rename(columns={"score": "score_cnn"}), on=["case", "service"])
ens["score"] = ens.groupby("case")["score_xgb"].rank(pct=True) + ens.groupby("case")["score_cnn"].rank(pct=True)
evaluate(ens).round(3)

,AC@1,AC@3,AC@5,Avg@5,MRR
all,0.930,0.993,0.996,0.978,0.960
ob,0.856,0.989,1.000,0.960,0.919
ss,0.944,0.989,0.989,0.976,0.965
tt,0.989,1.000,1.000,0.998,0.994


## Unseen system (leave one system out)

In [6]:
cnn_loso = pd.concat([run(tr, te) for _, tr, te in leave_one_system_out(idx)])
evaluate(cnn_loso).round(3)

,AC@1,AC@3,AC@5,Avg@5,MRR
all,0.859,0.930,0.989,0.931,0.906
ob,0.800,0.900,0.978,0.902,0.868
ss,0.922,0.956,1.000,0.962,0.948
tt,0.856,0.933,0.989,0.929,0.902


In [7]:
out = {"device": DEVICE, "epochs": EPOCHS, "seeds": list(SEEDS),
       "cnn_grouped_cv": evaluate(cnn_cv).round(4).to_dict(orient="index"),
       "ensemble_xgb_cnn": evaluate(ens).round(4).to_dict(orient="index"),
       "cnn_leave_one_system_out": evaluate(cnn_loso).round(4).to_dict(orient="index"),
       "bootstrap_cnn_minus_xgb": {m: list(map(float, bootstrap_diff(r_cnn, r_xgb, m))) for m in ["AC@1", "Avg@5", "MRR"]}}
json.dump(out, open(RESULTS / "deep_learning.json", "w"), indent=1)
cnn_cv.to_parquet(ROOT / "data" / "processed" / "oof_cnn.parquet", index=False)
print("saved reports/results/deep_learning.json")

saved reports/results/deep_learning.json


### Takeaways
- **The CNN is as good as XGBoost inside a system:** AC@1 0.930 vs 0.941 (same split). The difference is −0.011 with 95% CI [−0.037, +0.015], so no real difference. It needs no hand-made features.
- **The ensemble doesn't help** (AC@1 0.930, AC@3 0.993): averaging the two scores doesn't add accuracy at the top.
- **The CNN transfers better to a new system than XGBoost on all sources** (0.859 vs 0.744). It reads metrics only, so it has no trace problem. It is level with metrics-only XGBoost (0.878).
- **Cost:** ~17 minutes of A100 time for the grouped CV, vs seconds for XGBoost. For this data size, the gradient-boosted ranker is the practical choice; the CNN is the evidence that the signal is in the raw series.